# 1. Business Scenario:
A Bank has a portfolio of 50000 loans distributed across 10 economic sectors.<br>
Now, the **Risk Management Department** wants to:

**1.** Identify borrowers with a high probability of default.<br>
**2.** Estimate potential Financial Losses.<br>
**3.** Understand which industries carry the highest Credit Risk.<br>
**4.** Identify factors influencing borrowers credit ratings.<br>
**5.** Evaluate how the economic stress affects the loan portfolio.<br>
**6.** Develop a Machine Learning-based Credit Risk monitoring system.<br>

# 2. Understanding the Dataset

| File | Description | Purpose |
|---|---|---|
| loan_portfolio.csv | 50,000 loans, 24 attributes | Main modelling dataset |
| credit_ratings.csv | Annual credit-rating migrations | Rating transition analysis |
| macro_stress_scenarios.csv | Macroeconomic stress scenarios | Stress testing |
| portfolio_metrics.csv | Monthly portfolio risk metrics | Portfolio monitoring |
| vintage_analysis.csv | Cumulative default curves | Vintage analysis |
| credit_risk_column_descriptions.txt | Column descriptions | Data dictionary |

**Project Architecture**

             RAW DATA
                 |
                 |
       ---------------------
       |    |    |    |    |
      Loan Ratings Macro Portfolio Vintage
      Data  Data   Data  Data   Data
       |    |    |    |    |
       ---------------------
                 |
                 |
          DATA PROFILING
                 |
                 |
          DATA CLEANING
                 |
                 |
           DATA MERGING
                 |
                 |
          FEATURE ENGINEERING
                 |
                 |
        -----------------------
        |          |          |
       EDA       PD Model   LGD/EAD
        |          |        Models
        |          |          |
        -----------------------
                 |
                 |
          MODEL EVALUATION
                 |
                 |
         MODEL INTERPRETATION
                 |
                 |
          CREDIT RISK ENGINE
                 |
                 |
          BUSINESS INSIGHTS
                 |
                 |
          DASHBOARD / REPORT

**Project Folder Structure**

# **4. Project Setup and Loading the dataset**

## 4.A. Importing Libraries

In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical Analysis
from scipy import stats

# Machine Learning
from sklearn.model_selection import(
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import(
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import(
    roc_auc_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import(
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Model Interpretation
import shap

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Istallation of additional libraries if needed
!pip install xgboost lightgbm catboost shap

## 4.B.1> Loading the Dataset

In [3]:
# naming each dataset

ratings = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/credit_ratings.csv")
loan = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/loan_portfolio.csv")
macro = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/macro_stress_scenarios.csv")
payment = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/payment_history.csv")
portfolio = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/portfolio_metrics.csv")
vintage = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/vintage_analysis.csv")


## 4.B.2> Understanding the dataset<br>
Understanding the structure of the dataset before merging

In [4]:


datasets = {
    "ratings": ratings,
    "loan": loan,
    "macro": macro,
    "payment": payment,
    "portfolio": portfolio,
    "vintage": vintage
}
for name, df in datasets.items():
    print("\n", "="*60)
    print("DATASET:", name)
    
    print("Shape:", df.shape)  # to get shape i.e., rows,columns from each dataset
    
    print("\nColumns:")
    print(df.columns.tolist())  # to get columns from each dataset
    
    print("\nData Types:")
    print(df.dtypes)  #  to get dtypes from each dataset

    print("\nMissing Values:")
    print(df.isnull().sum())  # to get the total null values from each dataset

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())  # to get the duplicate values from each dataset




DATASET: ratings
Shape: (17939, 9)

Columns:
['issuer_id', 'sector', 'year', 'from_rating', 'to_rating', 'upgraded', 'downgraded', 'defaulted', 'notches_moved']

Data Types:
issuer_id        object
sector           object
year              int64
from_rating      object
to_rating        object
upgraded          int64
downgraded        int64
defaulted         int64
notches_moved     int64
dtype: object

Missing Values:
issuer_id        0
sector           0
year             0
from_rating      0
to_rating        0
upgraded         0
downgraded       0
defaulted        0
notches_moved    0
dtype: int64

Duplicate Rows:
0

DATASET: loan
Shape: (50000, 24)

Columns:
['loan_id', 'origination_date', 'maturity_date', 'maturity_months', 'sector', 'loan_type', 'collateral', 'initial_rating', 'credit_score', 'ead', 'coupon_rate', 'leverage', 'interest_coverage', 'debt_to_equity', 'pd_annual', 'lgd', 'el', 'unexpected_loss', 'rwa', 'defaulted', 'default_date', 'survival_months', 'recovery_rate', 'l

**Checking the complete data types**

In [5]:
print("\nComplete Data types of loan dataset:")
loan.info()

print("\nComplete Data types of ratings dataset:")
ratings.info()

print("\nComplete Data types of macro dataset:")
macro.info()

print("\nComplete Data types of payment dataset:")
payment.info()


Complete Data types of loan dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   loan_id             50000 non-null  object 
 1   origination_date    50000 non-null  object 
 2   maturity_date       50000 non-null  object 
 3   maturity_months     50000 non-null  int64  
 4   sector              50000 non-null  object 
 5   loan_type           50000 non-null  object 
 6   collateral          50000 non-null  object 
 7   initial_rating      50000 non-null  object 
 8   credit_score        50000 non-null  int64  
 9   ead                 50000 non-null  float64
 10  coupon_rate         50000 non-null  float64
 11  leverage            50000 non-null  float64
 12  interest_coverage   50000 non-null  float64
 13  debt_to_equity      50000 non-null  float64
 14  pd_annual           50000 non-null  float64
 15  lgd            

**Getting Summary Statistics**<br>
Here we get summary statistics for numerical columns

In [6]:
datasets = {
    "ratings": ratings,
    "loan": loan,
    "macro": macro,
    "payment": payment,
    "portfolio": portfolio,
    "vintage": vintage
}
for name, df in datasets.items():
    print("\n", "="*60)
    print("DATASET:", name)

    print("\nSummary Statistics:")
    print(df.describe().T)


DATASET: ratings

Summary Statistics:
                 count         mean       std     min     25%     50%     75%  \
year           17939.0  2019.309047  2.871778  2015.0  2017.0  2019.0  2022.0   
upgraded       17939.0     0.062044  0.241241     0.0     0.0     0.0     0.0   
downgraded     17939.0     0.069123  0.253670     0.0     0.0     0.0     0.0   
defaulted      17939.0     0.023245  0.150686     0.0     0.0     0.0     0.0   
notches_moved  17939.0     0.057082  0.552997    -6.0     0.0     0.0     0.0   

                  max  
year           2024.0  
upgraded          1.0  
downgraded        1.0  
defaulted         1.0  
notches_moved     7.0  

DATASET: loan

Summary Statistics:
                      count          mean           std           min  \
maturity_months     50000.0  5.397048e+01  2.671625e+01     12.000000   
credit_score        50000.0  7.149826e+02  5.089341e+01    518.000000   
ead                 50000.0  3.298605e+06  6.661943e+06  50000.000000   
co

## 4.C. Looking for Primary Keys

In [7]:
# to get unique values of the dataet

for name, df in datasets.items():
    print("\n", name)
    for col in df.columns:
        if "id" in col.lower():
            print(col, "Unique Values:", df[col].nunique())


 ratings
issuer_id Unique Values: 2000

 loan
loan_id Unique Values: 50000

 macro

 payment
loan_id Unique Values: 26322

 portfolio

 vintage


**Questions we need to answer**
**For the loan dataset:**

- Is each loan_id unique?
- Are there missing loan amounts?
- Are there duplicate loans?
- What are the numerical variables?
- What are the categorical variables?

**For the payment dataset:**

- Does each loan have multiple payment records?
- Are payment dates available?
- Is there a payment status?
- Are there missed payments?
- Are there partial payments?
- Is there an overdue amount?
These answers will determine our feature engineering strategy.

# **5. Merge the Datasets**

I would use star-schema inspired data architecture.<br>
**Proposed Structure:** <br>
**Fact Table:** *loan_portfolio*<br>

**Supporting Tables:** <br>
* Credit Ratings
* Macroeconomic scenarios
* Portfolio Metrics
* Vintage analysis

In [8]:
# Inspecting the loan and payment Dataset
# Display the columns

print(loan.head())
print(loan.columns.tolist())

print(payment.head())
print(payment.columns.tolist())

   loan_id origination_date maturity_date  maturity_months       sector  \
0  L000001       2017-10-01    2019-10-01               24   Technology   
1  L000002       2016-06-01    2020-06-01               48   Healthcare   
2  L000003       2023-08-01    2024-12-31               36  Real_Estate   
3  L000004       2016-04-01    2024-12-31              120  Real_Estate   
4  L000005       2015-10-01    2019-10-01               48       Energy   

   loan_type collateral initial_rating  credit_score         ead  ...  \
0   mortgage    secured              B           704  1035611.99  ...   
1  term_loan  unsecured              A           702   800786.92  ...   
2       bond    secured            BBB           699  4327864.12  ...   
3  term_loan    secured              B           641  3810461.19  ...   
4       bond  unsecured            CCC           597  2925538.53  ...   

   pd_annual     lgd         el  unexpected_loss         rwa  defaulted  \
0   0.041076  0.7858   33426.80    

In [9]:
# Identify the common loan identifier : (loan_id)
print(loan['loan_id'].nunique())
print(payment['loan_id'].nunique())

50000
26322


# Merging the datasets loan and payment